# MCC Data Prep (Webis → JSONL)

notebook `convert_webis_to_jsonl.py`  notebook 

## Steps
1. Run the path setup cell.
2. Configure input/output paths.
3. Run the conversion cell.

In [ ]:
import sys
from pathlib import Path

# Ensure project src/ is on the import path
project_root = Path.cwd().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Using project root: {project_root}")
print(f"Added to sys.path: {src_path}")

In [ ]:
from pathlib import Path

# === Configuration ===
INPUT_DIR = Path("src/corpus-webis-editorials-16/annotated-txt/split-by-portal-final")
OUTPUT_JSONL = Path("src/data/editorials.jsonl")

In [ ]:
from scripts.convert_webis_to_jsonl import convert

convert(INPUT_DIR, OUTPUT_JSONL)
print(f"Wrote JSONL to {OUTPUT_JSONL}")

## Split into Train/Dev Sets

Now split the editorials.jsonl into mcc_train.jsonl and mcc_dev.jsonl

In [ ]:
import json
import random

# Read all examples
with open(OUTPUT_JSONL, 'r', encoding='utf-8') as f:
    all_data = [json.loads(line) for line in f if line.strip()]

# Shuffle and split (80% train, 20% dev)
random.seed(42)
random.shuffle(all_data)

split_idx = int(len(all_data) * 0.8)
train_data = all_data[:split_idx]
dev_data = all_data[split_idx:]

# Write splits
TRAIN_PATH = Path("src/data/mcc_train.jsonl")
DEV_PATH = Path("src/data/mcc_dev.jsonl")

with open(TRAIN_PATH, 'w', encoding='utf-8') as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

with open(DEV_PATH, 'w', encoding='utf-8') as f:
    for item in dev_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"Total examples: {len(all_data)}")
print(f"Train: {len(train_data)} → {TRAIN_PATH}")
print(f"Dev: {len(dev_data)} → {DEV_PATH}")

# Show label distribution
from collections import Counter
train_labels = Counter(item['label'] for item in train_data)
dev_labels = Counter(item['label'] for item in dev_data)
print(f"\nTrain labels: {dict(train_labels)}")
print(f"Dev labels: {dict(dev_labels)}")